# Stochastic Processes (Concept 31)

A **stochastic process** is an indexed family of random variables $\{X_t\}_{t \in T}$.
Here we explore the canonical discrete-time example: the **simple symmetric random walk**
$S_n = X_1 + \cdots + X_n$, where $X_i \in \{-1, +1\}$ are iid with probability $1/2$ each.

Goals of this notebook:
- visualise sample paths,
- empirically verify $E[S_n] = 0$ and $\mathrm{Var}(S_n) = n$,
- check the diffusive scaling $\mathrm{SD}(S_n) \sim \sqrt{n}$.

Stdlib only (`random`, `math`, `statistics`).

In [ ]:
import random
import math
import statistics

rng = random.Random(0)

def random_walk(n_steps, rng):
    path = [0]
    s = 0
    for _ in range(n_steps):
        s += 1 if rng.random() < 0.5 else -1
        path.append(s)
    return path

## 1. Five sample paths
Each sample path $n \mapsto S_n(\omega)$ is one *realization* of the process. Different
$\omega$ give different functions of $n$ — a stochastic process is a **random function**.

In [ ]:
n_steps = 100
paths = [random_walk(n_steps, rng) for _ in range(5)]
for k, p in enumerate(paths):
    head = ', '.join(str(x) for x in p[:10])
    print(f'path {k+1}: [{head}, ...]   S_{n_steps} = {p[-1]:+d}')

## 2. Empirical mean and variance of $S_n$
By independence and the variance-of-sum identity, theory predicts
$E[S_n] = 0$ and $\mathrm{Var}(S_n) = n$. We verify with a Monte Carlo over $10^4$ walks.

In [ ]:
n_walks = 10_000
rng2 = random.Random(42)
endpoints = []
for _ in range(n_walks):
    s = 0
    for _ in range(n_steps):
        s += 1 if rng2.random() < 0.5 else -1
    endpoints.append(s)

mean_emp = statistics.fmean(endpoints)
var_emp = statistics.pvariance(endpoints)
print(f'E[S_n]  empirical = {mean_emp:+.4f}   theory = 0')
print(f'Var(S_n) empirical = {var_emp:.4f}   theory = {n_steps}')
print(f'SD(S_n) empirical = {var_emp**0.5:.4f}   theory = {math.sqrt(n_steps):.4f}')

## 3. Diffusive scaling: variance grows linearly with $n$
We sweep over several walk lengths and confirm that $\mathrm{Var}(S_n)$ grows like $n$
(equivalently, $\mathrm{SD}(S_n)$ grows like $\sqrt{n}$).
This scaling is the seed of Brownian motion (concept 32).

In [ ]:
rng3 = random.Random(7)
for n in [10, 50, 100, 200, 500]:
    ends = []
    for _ in range(5_000):
        s = 0
        for _ in range(n):
            s += 1 if rng3.random() < 0.5 else -1
        ends.append(s)
    v = statistics.pvariance(ends)
    print(f'n = {n:4d}   Var(S_n) ~ {v:7.2f}   ratio v/n = {v/n:.3f}')

## 4. Connection to ML

- **RL trajectories** are discrete-time stochastic processes; the Markov property of an MDP
  is exactly $P(s_{t+1} \mid s_0, \dots, s_t, a_t) = P(s_{t+1} \mid s_t, a_t)$.
- **SGD iterates** $\{\theta_t\}$ form a stochastic process driven by random mini-batches.
- **Diffusion / flow-matching** sampling trajectories $\{z_t\}_{t \in [0,1]}$ are
  continuous-time stochastic processes; the random walk above is the discrete analogue
  whose continuum limit (Donsker's theorem) is Brownian motion.